# Medical Image Classification with MedMNIST

This notebook demonstrates how to use **AUCMEDI** with the **MedMNIST** benchmark datasets.

MedMNIST is a collection of standardized biomedical image datasets for medical image analysis. We'll use **PathMNIST** (pathology images) to demonstrate AUCMEDI's three pillars: DataInterface, NeuralNetwork, and DataGenerator.

## Setup and Installation

First, install the medmnist package:

In [ ]:
!pip install medmnist
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0"

## Import Libraries

In [ ]:
import numpy as np
from medmnist import PathMNIST
import aucmedi
from aucmedi import *

## Load MedMNIST Data

We'll use PathMNIST, which contains 107,180 pathology images from 9 tissue classes.

In [ ]:
# Download and load PathMNIST
train_dataset = PathMNIST(split='train', download=True)
val_dataset = PathMNIST(split='val', download=True)
test_dataset = PathMNIST(split='test', download=True)

print(f'Train: {len(train_dataset)} samples')
print(f'Val: {len(val_dataset)} samples')
print(f'Test: {len(test_dataset)} samples')

## Convert to AUCMEDI Format

AUCMEDI expects data in a specific format. We'll convert the MedMNIST data:

In [ ]:
# Prepare data for AUCMEDI
# Create sample lists and labels
train_samples = []
train_labels = []
for i in range(len(train_dataset)):
    train_samples.append(train_dataset[i][0])
    train_labels.append(train_dataset[i][1])

test_samples = []
test_labels = []
for i in range(len(test_dataset)):
    test_samples.append(test_dataset[i][0])
    test_labels.append(test_dataset[i][1])

# Convert to numpy arrays
train_x = np.array([np.array(img) for img in train_samples])
train_y = np.array(train_labels).flatten()
test_x = np.array([np.array(img) for img in test_samples])
test_y = np.array(test_labels).flatten()

print(f'Train images shape: {train_x.shape}')
print(f'Train labels shape: {train_y.shape}')
print(f'Number of classes: {len(np.unique(train_y))}')

## AUCMEDI's Three Pillars

AUCMEDI is built on three pillars:
1. **DataInterface**: Handles data loading
2. **NeuralNetwork**: Manages the model architecture
3. **DataGenerator**: Provides data augmentation and preprocessing

### 1. DataInterface Setup

In [ ]:
# Create index lists
train_samples_idx = list(range(len(train_x)))
test_samples_idx = list(range(len(test_x)))

# Initialize DataInterface for in-memory data
ds = DataInterface(interface="numpy",
                   data_directory=None,
                   training_samples=train_samples_idx,
                   validation_samples=None,
                   test_samples=test_samples_idx)

print('DataInterface initialized')

### 2. NeuralNetwork Setup

We'll use a DenseNet121 architecture pretrained on ImageNet:

In [ ]:
# Define model
model = NeuralNetwork(n_labels=9,
                      channels=3,
                      architecture="DenseNet121",
                      pretrained_weights=True,
                      loss="categorical_crossentropy",
                      metrics=["accuracy"])

print('Model architecture:', model.architecture)
print('Input shape:', (28, 28, 3))

### 3. DataGenerator Setup

Configure data augmentation and preprocessing:

In [ ]:
from aucmedi.data_processing.subfunctions import Resize

# Create subfunctions for preprocessing
# Resize images to 224x224 for DenseNet
sf_list = [Resize(shape=(224, 224))]

# Initialize DataGenerators
train_gen = DataGenerator(train_samples_idx,
                          path_imagedir=None,
                          labels=train_y,
                          image_format="array",
                          batch_size=32,
                          data_aug=None,
                          shuffle=True,
                          subfunctions=sf_list,
                          resize=None,
                          standardize_mode="tf",
                          grayscale=False,
                          sample_weights=None,
                          seed=None,
                          image_loader=lambda idx: train_x[idx])

test_gen = DataGenerator(test_samples_idx,
                         path_imagedir=None,
                         labels=test_y,
                         image_format="array",
                         batch_size=32,
                         data_aug=None,
                         shuffle=False,
                         subfunctions=sf_list,
                         resize=None,
                         standardize_mode="tf",
                         grayscale=False,
                         sample_weights=None,
                         seed=None,
                         image_loader=lambda idx: test_x[idx])

print('DataGenerators initialized')

## Training

Train the model on PathMNIST data:

In [ ]:
# Train model
history = model.train(train_gen,
                      epochs=10,
                      validation_freq=1,
                      callbacks=[])

print('Training complete!')

## Evaluation

Evaluate the model on the test set:

In [ ]:
# Predict on test set
preds = model.predict(test_gen)

# Calculate accuracy
from sklearn.metrics import accuracy_score, classification_report

pred_labels = np.argmax(preds, axis=1)
accuracy = accuracy_score(test_y, pred_labels)

print(f'Test Accuracy: {accuracy:.4f}')
print('\nClassification Report:')
print(classification_report(test_y, pred_labels))

## Visualization

Visualize some predictions:

In [ ]:
import matplotlib.pyplot as plt

# Show sample predictions
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.ravel()

class_names = ['ADI', 'BACK', 'DEB', 'LYM', 'MUC', 'MUS', 'NORM', 'STR', 'TUM']

for i in range(10):
    axes[i].imshow(test_x[i])
    axes[i].set_title(f'True: {class_names[test_y[i]]}\nPred: {class_names[pred_labels[i]]}')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## Summary

This tutorial demonstrated:
- Loading MedMNIST datasets
- Converting to AUCMEDI format
- Using AUCMEDI's three pillars (DataInterface, NeuralNetwork, DataGenerator)
- Training and evaluating on medical imaging data

Try experimenting with:
- Different MedMNIST datasets (ChestMNIST, DermaMNIST, etc.)
- Different architectures (ResNet, EfficientNet, etc.)
- Data augmentation techniques
- Ensemble methods